# 02 — Identity Lifecycle

The Wave-1 identity layer (ADR 0009) lifts specter-1 above the "frozen roster at swarm formation" assumption. Four composable pieces, all drop-in at the bus boundary:

* **`MutableRoster`** — timestamped add/remove/rotate + audit log
* **`KeyRotationAnnouncement`** — sign-the-new-key-with-the-old-key rotation
* **`RevocationList`** — forward-only key blocklist
* **`AttestationProvider`** — TPM/Secure-Enclave gate (mock impl ships)

## Operator question

**What stops the adversary from spinning up fake drones?**

Before the trust math runs, before reputation accrues — the *identity*
layer is the bouncer. A drone with no roster entry can't sign a valid
envelope; a revoked drone can be silenced mid-mission; a rotated key
keeps the same agent ID continuous across compromise recovery. This
notebook walks through the lifecycle every agent passes through.

In the React Workshop Console, Lesson 07 (`sybil_cabal`) is where this
defense visibly fires — two phantom radios outside the AO try to vouch
for each other, and V3 transitive presence rejects them. The math
notebook 04 derives is what makes that scene work. Here we cover the
gate behind it.


In [1]:
from specter.crypto import Keypair
from specter.identity import (
    KeyRotationAnnouncement, MockAttestationProvider, MutableRoster,
    RevocationList, apply_rotation, sign_rotation, verify_rotation,
)
from specter.messages import KIND_POSE, PoseReport, encode
from specter.secure_bus import Identity, ReplayWindow, VerificationError

## Intuition: rotate a key, watch the old envelopes get rejected as `key_revoked_post_rotation`

Standard rotation: the new key is announced via an envelope **signed by the old key**. After `apply_rotation`, the roster believes the new key. Any straggling envelope signed with the old key is rejected with a forensically-distinct category — *not* `bad_signature` — so operators can tell stale-sender bugs from active forgery.

In [2]:
alpha = Identity('alpha', Keypair.generate())
roster = MutableRoster()
roster.add_peer('alpha', alpha.keypair.public_bytes, t_ns=0)

# Pre-rotation pose verifies fine.
pose = PoseReport('alpha', x=1.0, y=2.0, theta=0.0, timestamp_ns=1_000_000_000)
old_env = alpha.seal(KIND_POSE, encode(pose), timestamp_ns=1_000_000_000)
_ = roster.verify_envelope(old_env, ReplayWindow())
print('pre-rotation: verified')

# Rotate.
new_kp = Keypair.generate()
ann = KeyRotationAnnouncement(
    agent_id='alpha',
    old_pubkey=alpha.keypair.public_bytes,
    new_pubkey=new_kp.public_bytes,
    effective_t_ns=2_000_000_000,
)
old_signature = sign_rotation(ann, alpha.keypair)
assert verify_rotation(ann, old_signature, t_ns=2_000_000_000)
apply_rotation(roster, ann, t_ns=2_000_000_000)

# An envelope signed by the OLD key, dated AFTER the rotation, is now stale.
stale_env = alpha.seal(KIND_POSE, encode(pose), timestamp_ns=3_000_000_000)
try:
    roster.verify_envelope(stale_env, ReplayWindow())
except VerificationError as e:
    print(f'post-rotation old-key envelope rejected: {e}')

# Same identity sealing with the NEW key works.
alpha = Identity('alpha', new_kp)
fresh_env = alpha.seal(KIND_POSE, encode(pose), timestamp_ns=4_000_000_000)
roster.verify_envelope(fresh_env, ReplayWindow())
print('post-rotation new-key envelope: verified')

pre-rotation: verified
post-rotation old-key envelope rejected: key_revoked_post_rotation: alpha signed with rotated-out key
post-rotation new-key envelope: verified


## Intuition: revocation is a forward-only blocklist

`RevocationList` is keyed on raw pubkey bytes (the same X9.62 form `Sros2Bus` uses). Once you revoke at time T, every envelope timestamped ≥ T from that key is rejected with `key_revoked`. There is no un-revoke.

In [3]:
rev = RevocationList()
victim = Keypair.generate()
rev.revoke(victim.public_bytes, t_ns=5_000_000_000)
print('before revocation time:', rev.is_revoked(victim.public_bytes, t_ns=4_999_999_999))
print('at revocation time:    ', rev.is_revoked(victim.public_bytes, t_ns=5_000_000_000))
print('after revocation time: ', rev.is_revoked(victim.public_bytes, t_ns=6_000_000_000))

before revocation time: False
at revocation time:     True
after revocation time:  True


## Claim: attestation blocks sybil-mint by an honest-but-compromised robot

Attestation simulates TPM/Secure-Enclave: only keys in the allowlist are accepted, regardless of roster membership. A real attacker who steals a hardware key off Robot A *cannot* mint a new sybil key from Robot B because the new key won't pass attestation.

**Audited by:** `tests/eval/test_identity_attacks.py` — sybil-mint blocked when attestation is required.

In [4]:
honest_pubs = {Keypair.generate().public_bytes for _ in range(3)}
attestation = MockAttestationProvider(allowlist=honest_pubs)
rogue = Keypair.generate()

for pub in list(honest_pubs)[:1]:
    print(f'honest key attested? {attestation.is_attested(pub)}')
print(f'rogue key attested?  {attestation.is_attested(rogue.public_bytes)}')
assert not attestation.is_attested(rogue.public_bytes)

honest key attested? True
rogue key attested?  False


## Claim: key compromise without rotation discipline = wire-layer drop

Coverage Parity Wave 4. `single_bad_key` runs alpha through a mid-mission
key swap — alpha's keypair changes but its roster entry doesn't, so every
envelope it now signs fails ECDSA verification at the receivers. The trust
evaluator never sees the payload; reputation stays near the uniform prior
for every real-agent pair. The defense is the same envelope-boundary
discipline taught at the very top of this notebook — without
`KeyRotationAnnouncement` to update the roster, the agent is simply gone
from the network. Pairs with rail Lesson 13 (`bad_key`).

Claim source: `tests/eval/test_attack_battery.py::test_single_bad_key_collapses_immediately`.


In [5]:
import os, sys
from pathlib import Path
_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from tests.eval.runner import run_scenario
from tests.eval import scenarios

result = run_scenario(scenarios.single_bad_key())
print('alpha keypair swapped at t=1; envelopes fail signature verification thereafter.')
print()
print('Per-viewer reputation of alpha (the swapped-key agent):')
for viewer, peers in result.final_rep.items():
    if viewer == 'alpha':
        continue
    print(f'  {viewer} -> alpha = {peers["alpha"]:.3f}')
print()
print('Trust evaluator never moves these reps because rejected envelopes')
print('drop at the bus boundary, before the cohort math runs.')


alpha keypair swapped at t=1; envelopes fail signature verification thereafter.

Per-viewer reputation of alpha (the swapped-key agent):
  bravo -> alpha = 0.004
  charlie -> alpha = 0.004
  delta -> alpha = 0.004

Trust evaluator never moves these reps because rejected envelopes
drop at the bus boundary, before the cohort math runs.


## Limit

The attestation provider shipped here (`MockAttestationProvider`) is an allowlist — fine for tests, demos, and notebooks, but **not** a real defense. In a real deployment the attestation step talks to TPM 2.0 / Secure Enclave / Apple Secure Enclave / etc. ADR 0009 records this; integration is Phase 4 per `docs/HARDWARE_READINESS.md`.

Today's residual: a thief with physical access to a robot's storage can extract the signing key. With attestation enabled, that key still works *on that robot*; the defense kicks in when the thief tries to mint **new** keys. Hardware-key-compromise on a single device remains a Phase-4 problem.